# Spanish → Asturian MT (evaluation)

This notebook loads a **fine-tuned** model from Google Drive, evaluates it on FLORES+ devtest, and saves the translations to a text file.

- **Baseline checkpoint**: `Helsinki-NLP/opus-mt-es-ca`
- **Fine-tuned checkpoint**: loaded from Drive
- **Eval**: FLORES+ `devtest` (`spa_Latn` → `ast_Latn`)

**Outputs**

- `flores_devtest_hypotheses_ast.txt` (one hypothesis per line)
- BLEU scores printed in the notebook


### 0. Setup (Colab)

Note: the model was trained locally. This notebook is only for loading it from Drive and running the evaluation.


In [5]:
from pathlib import Path


def find_project_dir() -> Path:
    # If I run from the project folder, this works.
    cwd = Path.cwd().resolve()
    if (cwd / "finetuned_es_ast").is_dir():
        return cwd

    # Otherwise, walk up a few levels.
    for p in [cwd] + list(cwd.parents)[:4]:
        if (p / "finetuned_es_ast").is_dir():
            return p

    # Colab default Drive location (if running in Colab).
    drive_guess = Path("/content/drive/MyDrive/MT/Aus-Spa MT/spanish_austurian_mt")
    if (drive_guess / "finetuned_es_ast").is_dir():
        return drive_guess

    return cwd


# Mount Drive only if running in Colab
try:
    from google.colab import drive

    drive.mount("/content/drive")
except Exception:
    pass

PROJECT_DIR = find_project_dir()
FINETUNED_DIR = (PROJECT_DIR / "finetuned_es_ast").resolve()

print("PROJECT_DIR:", PROJECT_DIR)
print("FINETUNED_DIR:", FINETUNED_DIR)
assert FINETUNED_DIR.is_dir(), "I can't find finetuned_es_ast/ next to this notebook."


PROJECT_DIR: /home/drk/Desktop/Masters/MT/Coursework/labs/Shared_task_2/spanish_asturian_mt
FINETUNED_DIR: /home/drk/Desktop/Masters/MT/Coursework/labs/Shared_task_2/spanish_asturian_mt/finetuned_es_ast


### 1. Install dependencies

Run once per Colab session.

In [6]:
%pip install -q "transformers>=4.40.0" "datasets>=2.14.0" evaluate sacrebleu sentencepiece torch


Note: you may need to restart the kernel to use updated packages.


### 2. Hugging Face token (for FLORES+)

If I run this locally, I set it in my shell:

`export HF_TOKEN=...`

If I run this in Colab, I add a Secret named `mt` (or `HF_TOKEN`).

### 3. Load the fine-tuned model from Drive

This loads the model you trained locally and does a quick sanity-check translation.

In [7]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

print('GPU available:', torch.cuda.is_available())

tokenizer = AutoTokenizer.from_pretrained(str(FINETUNED_DIR))
model = AutoModelForSeq2SeqLM.from_pretrained(str(FINETUNED_DIR))

# Quick sanity check
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device).eval()

test_sentences = [
    'Faltan pocos días para el comienzo de las clases en los colegios de Primaria.',
    'La investigación todavía se ubica en su etapa inicial.',
]

enc = tokenizer(test_sentences, return_tensors='pt', padding=True, truncation=True, max_length=128).to(device)
with torch.inference_mode():
    out = model.generate(**enc, num_beams=4, max_length=128)

preds = tokenizer.batch_decode(out, skip_special_tokens=True)
for s, p in zip(test_sentences, preds):
    print('-' * 70)
    print('SRC:', s)
    print('PRED:', p)


GPU available: True


/home/drk/.local/lib/python3.14/site-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


----------------------------------------------------------------------
SRC: Faltan pocos días para el comienzo de las clases en los colegios de Primaria.
PRED: Falten pocos díes pal empiezu de les clases nos colexos de Primaria.
----------------------------------------------------------------------
SRC: La investigación todavía se ubica en su etapa inicial.
PRED: La investigación inda s'atopa na so etapa inicial.


### 4. FLORES+ evaluation (baseline vs fine-tuned)

This compares the baseline model against the fine-tuned model and saves the fine-tuned translations to a text file.

In [8]:
import os

import evaluate
import torch
from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Token: try (1) local env, (2) Colab secrets, (3) HF CLI login cache, (4) prompt once
hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")

# Colab secrets
if not hf_token:
    try:
        from google.colab import userdata

        hf_token = userdata.get("mt")
        if not hf_token:
            hf_token = userdata.get("HF_TOKEN")
    except Exception:
        pass

# Local cache from `huggingface-cli login`
if not hf_token:
    try:
        from huggingface_hub import HfFolder

        hf_token = HfFolder.get_token()
    except Exception:
        pass

# Last resort: prompt in-notebook
if not hf_token:
    try:
        import getpass

        hf_token = getpass.getpass("HF token (for FLORES+): ")
    except Exception:
        hf_token = None

assert hf_token, "I need a Hugging Face token to load FLORES+."
flores_spa = load_dataset("openlanguagedata/flores_plus", "spa_Latn", split="devtest", token=hf_token)
flores_ast = load_dataset("openlanguagedata/flores_plus", "ast_Latn", split="devtest", token=hf_token)

src = flores_spa["text"]
refs = [[t] for t in flores_ast["text"]]

metric = evaluate.load("sacrebleu")

def translate(src_sentences, tok, mdl, batch_size=16, num_beams=4, max_length=128):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    mdl = mdl.to(device).eval()
    out_text = []
    with torch.inference_mode():
        for i in range(0, len(src_sentences), batch_size):
            batch = src_sentences[i : i + batch_size]
            enc = tok(batch, return_tensors="pt", padding=True, truncation=True, max_length=max_length).to(device)
            out = mdl.generate(**enc, num_beams=num_beams, max_length=max_length)
            out_text.extend(tok.batch_decode(out, skip_special_tokens=True))
    return out_text

BASELINE = "Helsinki-NLP/opus-mt-es-ca"

# Baseline
b_tok = AutoTokenizer.from_pretrained(BASELINE)
b_mdl = AutoModelForSeq2SeqLM.from_pretrained(BASELINE)
b_preds = translate(src, b_tok, b_mdl)
b_bleu = metric.compute(predictions=b_preds, references=refs)["score"]
print(f"Baseline BLEU: {b_bleu:.2f}")

# Fine-tuned
ft_tok = AutoTokenizer.from_pretrained(str(FINETUNED_DIR))
ft_mdl = AutoModelForSeq2SeqLM.from_pretrained(str(FINETUNED_DIR))
ft_preds = translate(src, ft_tok, ft_mdl)
ft_bleu = metric.compute(predictions=ft_preds, references=refs)["score"]
print(f"Fine-tuned BLEU: {ft_bleu:.2f}")
print(f"Delta: {ft_bleu - b_bleu:+.2f}")

out_path = PROJECT_DIR / "flores_devtest_hypotheses_ast.txt"
out_path.write_text("\n".join(ft_preds) + "\n", encoding="utf-8")
print("Saved:", out_path)

for i in range(3):
    print("-" * 70)
    print("SRC:", src[i])
    print("REF:", flores_ast["text"][i])
    print("BASE:", b_preds[i])
    print("FT  :", ft_preds[i])


Resolving data files:   0%|          | 0/225 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/220 [00:00<?, ?it/s]

TypeError: Pickler._batch_setitems() takes 2 positional arguments but 3 were given